# Problema 2 — Estimación de Edad con CNNs
## Notebook 2 de 2: Entrenamiento

> ⚠️ **Prerequisito:** ejecuta primero `P2_01_dataloaders.ipynb` para generar `dataloaders_utk.pth`.

---

### ¿Qué haremos?

```
dataloaders_utk.pth
       │
       ▼
  Cargar DataLoaders         ← ya está dado
       │
       ▼
  Baseline: SimpleCNN        ← ya está dado (3 bloques, igual al ejemplo)
       │
       ▼
  🔧 MejorCNN                ← TU TAREA: agregar más capas
       │
       ▼
  Entrenar y comparar        ← ya está dado
```

---

### 📚 Antes de empezar: investiga CNNs

Responde estas preguntas en la celda de abajo **antes** de escribir código:

1. ¿Qué hace `nn.Conv2d(in_channels, out_channels, kernel_size, padding)`? ¿Qué es un *filtro*?
2. ¿Qué es un **feature map**? ¿Cuántos feature maps produce `Conv2d(3, 32, 3)`?
3. ¿Para qué sirve `MaxPool2d(2)`? ¿Qué le pasa al tamaño espacial H×W?
4. ¿Por qué las CNN funcionan mejor que los MLP para imágenes?

**Recursos recomendados:**
- [PyTorch `nn.Conv2d` docs](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [CS231n: Convolutional Neural Networks](https://cs231n.github.io/convolutional-networks/)
- [3Blue1Brown: But what is a convolution?](https://www.youtube.com/watch?v=KuXjwB4LzSA)

In [ ]:
# 📝 Escribe aquí tus respuestas:

# 1. Conv2d: ...
# 2. Feature map: ...
# 3. MaxPool2d: ...
# 4. CNN vs MLP en imágenes: ...

---
## 0. Importaciones

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")

---
## 1. Cargar los DataLoaders

In [ ]:
assert Path("dataloaders_utk.pth").exists(), \
    "❌ No encontré dataloaders_utk.pth — ejecuta primero P2_01_dataloaders.ipynb"

ckpt = torch.load("dataloaders_utk.pth", map_location=DEVICE)

train_loader   = ckpt["train_loader"]
val_loader     = ckpt["val_loader"]
test_loader    = ckpt["test_loader"]
IMG_SIZE       = ckpt["img_size"]        # 64
BATCH_SIZE     = ckpt["batch_size"]      # 32
IMAGENET_MEAN  = ckpt["imagenet_mean"]
IMAGENET_STD   = ckpt["imagenet_std"]

print("DataLoaders cargados ✓")
print(f"  train : {len(train_loader)} batches × {BATCH_SIZE}")
print(f"  val   : {len(val_loader)} batches")
print(f"  test  : {len(test_loader)} batches")

imgs, ages = next(iter(train_loader))
print(f"\nShapes: imgs={imgs.shape}, ages={ages.shape}")

---
## 2. Hiperparámetros

In [ ]:
NUM_EPOCHS = 15
LR         = 1e-3

print(f"Épocas : {NUM_EPOCHS}")
print(f"LR     : {LR}")
print(f"Imagen : {IMG_SIZE}×{IMG_SIZE} px, 3 canales")

---
## 3. Modelo Baseline: `SimpleCNN`

Este modelo es el punto de partida. Es idéntico al del ejemplo `age_regression_dataloader.ipynb`:
3 bloques convolucionales y una cabeza de regresión que predice un único valor (la edad).

```
[3, 64, 64]  → Conv(3→16)  → ReLU → MaxPool → [16, 32, 32]
             → Conv(16→32) → ReLU → MaxPool → [32, 16, 16]
             → Conv(32→64) → ReLU → MaxPool → [64,  8,  8]
             → Flatten                       → [4096]
             → Linear(4096→128) → ReLU
             → Linear(128→1)               ← edad predicha
```

In [ ]:
class SimpleCNN(nn.Module):
    """
    CNN baseline de 3 bloques para regresión de edad.
    Arquitectura intencionalmente simple — es el punto de partida.
    """

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Bloque 1: extrae bordes y texturas básicas
            nn.Conv2d(3, 16, kernel_size=3, padding=1),    # [3,64,64]  → [16,64,64]
            nn.ReLU(),
            nn.MaxPool2d(2),                               # [16,64,64] → [16,32,32]

            # Bloque 2: patrones de nivel medio
            nn.Conv2d(16, 32, kernel_size=3, padding=1),   # [16,32,32] → [32,32,32]
            nn.ReLU(),
            nn.MaxPool2d(2),                               # [32,32,32] → [32,16,16]

            # Bloque 3: características de alto nivel
            nn.Conv2d(32, 64, kernel_size=3, padding=1),   # [32,16,16] → [64,16,16]
            nn.ReLU(),
            nn.MaxPool2d(2),                               # [64,16,16] → [64, 8, 8]
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),                   # [64,8,8]  → [4096]
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 1),             # → predicción de edad
        )

    def forward(self, x):
        return self.regressor(self.features(x))


# ── Verificar shapes ──────────────────────────────────────────────────────────
_m = SimpleCNN().to(DEVICE)
with torch.inference_mode():
    _out = _m(torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(DEVICE))
assert _out.shape == (4, 1), f"Shape incorrecto: {_out.shape}"

n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"SimpleCNN — Parámetros: {n_params:,}")
print(f"Output shape: {_out.shape}   ← correcto: [batch, 1]")

---
## 4. Tu CNN mejorada: `MejorCNN`

### Cómo trazar las dimensiones

`Conv2d` con `padding=1` y `kernel_size=3` **no cambia** el tamaño espacial H×W.  
Solo `MaxPool2d(2)` divide H y W a la mitad.

Con imagen de entrada 64×64:

```
Input:         [B,  3, 64, 64]
Conv(3→32):    [B, 32, 64, 64]   ← solo cambia la profundidad (canales)
MaxPool:       [B, 32, 32, 32]   ← ÷2 en H y W
Conv(32→64):   [B, 64, 32, 32]
MaxPool:       [B, 64, 16, 16]
Conv(64→128):  [B, 128, 16, 16]
MaxPool:       [B, 128,  8,  8]
Conv(128→256): [B, 256,  8,  8]
MaxPool:       [B, 256,  4,  4]
Conv(256→256): [B, 256,  4,  4]
MaxPool:       [B, 256,  2,  2]   ← con 5 MaxPool llegas a 2×2
Flatten:       [B, 256*2*2=1024]
Linear(1024→256) → ReLU → Linear(256→1)
```

> ⚠️ **Cuándo parar:** con `MaxPool2d(2)` y una imagen de 64×64, después de 5 pools llegas a 2×2.  
> Agregar más pools daría 1×1 — en ese punto la información espacial se pierde completamente.

---

### 🔧 TODO 1 — Implementa `MejorCNN`

**Requisitos mínimos:**
1. Al menos **5 bloques convolucionales** (el baseline tiene 3 — agrega 2 más)
2. `BatchNorm2d` después de cada `Conv2d` (estabiliza el entrenamiento)
3. `Dropout` antes de la capa final del regresor (reduce overfitting)
4. El `forward` debe funcionar con imágenes de `[B, 3, 64, 64]` y retornar `[B, 1]`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 🔧 TODO 1: Implementa MejorCNN
# ─────────────────────────────────────────────────────────────────────────────
class MejorCNN(nn.Module):
    """
    Tu CNN mejorada para regresión de edad.

    Requisitos:
      - Mínimo 5 bloques conv (el baseline tiene 3)
      - BatchNorm2d después de cada Conv2d
      - Dropout antes de la capa de salida
      - Retorna tensor de shape [B, 1]

    Plantilla (completa los ???):

      [B, 3, 64, 64]
        → Conv(3→32)   + BN + ReLU + MaxPool  → [B, 32, 32, 32]
        → Conv(32→64)  + BN + ReLU + MaxPool  → [B, 64, 16, 16]
        → Conv(64→128) + BN + ReLU + MaxPool  → [B, 128, 8, 8]
        → Conv(??→???)  + BN + ReLU + MaxPool → [B, ???, 4, 4]
        → Conv(??→???)  + BN + ReLU + MaxPool → [B, ???, 2, 2]
        → Flatten                             → [B, ???]
        → Linear(???, 256) → ReLU → Dropout
        → Linear(256, 1)
    """

    def __init__(self):
        super().__init__()

        # ── Bloques convolucionales ────────────────────────────────────────────
        # Cada bloque: Conv2d → BatchNorm2d → ReLU → MaxPool2d
        #
        # Nota sobre BatchNorm2d:
        #   nn.BatchNorm2d(num_features)  ← num_features = out_channels de la Conv anterior
        #   Normaliza los feature maps por batch. Permite usar learning rates más altos
        #   y hace el entrenamiento más estable.

        self.features = nn.Sequential(
            # --- TU CÓDIGO AQUÍ ---
            # Bloque 1: [B, 3, 64, 64] → [B, 32, 32, 32]
            # nn.Conv2d(3, 32, kernel_size=3, padding=1),
            # nn.BatchNorm2d(32),
            # nn.ReLU(),
            # nn.MaxPool2d(2),

            # Bloque 2: [B, 32, 32, 32] → [B, 64, 16, 16]
            # ...

            # Bloque 3: [B, 64, 16, 16] → [B, 128, 8, 8]
            # ...

            # Bloque 4 (NUEVO): → [B, ???, 4, 4]
            # ...

            # Bloque 5 (NUEVO): → [B, ???, 2, 2]
            # ...
            # ----------------------
        )

        # ── Cabeza de regresión ────────────────────────────────────────────────
        # CUIDADO: el tamaño del primer Linear depende de tu arquitectura.
        # Calcula: out_channels_último_bloque × H_final × W_final
        # Con 5 MaxPool sobre entrada 64×64: H_final = W_final = 64 / 2^5 = 2

        self.regressor = nn.Sequential(
            # --- TU CÓDIGO AQUÍ ---
            # nn.Flatten(),
            # nn.Linear(??? * 2 * 2, 256),
            # nn.ReLU(),
            # nn.Dropout(0.4),
            # nn.Linear(256, 1),
            # ----------------------
        )

    def forward(self, x):
        """
        x : [B, 3, 64, 64]
        retorna: [B, 1]
        """
        # --- TU CÓDIGO AQUÍ ---
        raise NotImplementedError("Implementa el forward")
        # ----------------------


# ── Test de shapes ─────────────────────────────────────────────────────────────
mi_cnn = MejorCNN().to(DEVICE)

with torch.inference_mode():
    dummy = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out = mi_cnn(dummy)
    assert out.shape == (4, 1), f"Shape incorrecto: {out.shape}, esperado (4, 1)"
    print(f"✅ Forward pass OK: {dummy.shape} → {out.shape}")

n_params = sum(p.numel() for p in mi_cnn.parameters() if p.requires_grad)
print(f"   Parámetros: {n_params:,}")
print(mi_cnn)

---
## 5. Funciones de entrenamiento y evaluación

El loop es el mismo para ambos modelos. Usamos:
- **Loss:** `MSELoss` — penaliza errores cuadráticos (para backprop)
- **Métrica:** `MAE` — error medio en años (interpretable)

In [ ]:
def compute_mae(preds: torch.Tensor, targets: torch.Tensor) -> float:
    """Mean Absolute Error — interpretable en 'años'."""
    return torch.abs(preds - targets).mean().item()


def train_model(model, train_loader, val_loader,
                num_epochs=NUM_EPOCHS, lr=LR,
                device=DEVICE, model_name="modelo"):
    """
    Entrena un modelo de regresión de edad.

    Returns:
        history : dict con listas train_loss, val_loss, train_mae, val_mae
    """
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    history = {"train_loss": [], "val_loss": [], "train_mae": [], "val_mae": []}
    best_val_mae = float("inf")

    print(f"\n{'Época':>5} | {'Train Loss':>10} | {'Train MAE':>9} | {'Val Loss':>8} | {'Val MAE':>7}")
    print("-" * 55)

    for epoch in range(1, num_epochs + 1):

        # ── TRAIN ─────────────────────────────────────────────────────────────
        model.train()
        tl, tm = 0.0, 0.0

        for imgs, ages in train_loader:
            imgs, ages = imgs.to(device), ages.to(device)
            optimizer.zero_grad()
            preds = model(imgs).squeeze(1)  # [B,1] → [B]
            loss = criterion(preds, ages)
            loss.backward()
            optimizer.step()
            tl += loss.item()
            tm += compute_mae(preds, ages)

        train_loss = tl / len(train_loader)
        train_mae  = tm / len(train_loader)

        # ── VALIDACIÓN ────────────────────────────────────────────────────────
        model.eval()
        vl, vm = 0.0, 0.0

        with torch.inference_mode():
            for imgs, ages in val_loader:
                imgs, ages = imgs.to(device), ages.to(device)
                preds = model(imgs).squeeze(1)
                vl += criterion(preds, ages).item()
                vm += compute_mae(preds, ages)

        val_loss = vl / len(val_loader)
        val_mae  = vm / len(val_loader)

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_mae"].append(train_mae)
        history["val_mae"].append(val_mae)

        marker = " ← mejor" if val_mae < best_val_mae else ""
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            torch.save(model.state_dict(), f"mejor_{model_name}.pth")

        print(f"{epoch:>5} | {train_loss:>10.2f} | {train_mae:>9.2f} | {val_loss:>8.2f} | {val_mae:>7.2f}{marker}")

    print(f"\nMejor Val MAE: {best_val_mae:.2f} años")
    return history


print("✅ Funciones de entrenamiento definidas")

---
## 6. Entrenar el Baseline (SimpleCNN)

In [ ]:
baseline = SimpleCNN().to(DEVICE)
n = sum(p.numel() for p in baseline.parameters() if p.requires_grad)
print(f"Entrenando SimpleCNN ({n:,} parámetros)...")
history_base = train_model(baseline, train_loader, val_loader, model_name="simple")

---
## 7. Entrenar tu CNN mejorada

In [ ]:
mi_cnn = MejorCNN().to(DEVICE)
n = sum(p.numel() for p in mi_cnn.parameters() if p.requires_grad)
print(f"Entrenando MejorCNN ({n:,} parámetros)...")
history_mejor = train_model(mi_cnn, train_loader, val_loader, model_name="mejor")

---
## 8. Comparación de curvas de aprendizaje

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("SimpleCNN vs MejorCNN — Curvas de Aprendizaje", fontsize=13)

# MSE Loss
ax1.plot(epochs_range, history_base["val_loss"],  label="SimpleCNN val",  linestyle="--")
ax1.plot(epochs_range, history_mejor["val_loss"], label="MejorCNN val",   linestyle="-")
ax1.set_title("Val Loss (MSE)"); ax1.set_xlabel("Época"); ax1.set_ylabel("MSE")
ax1.legend(); ax1.grid(True, alpha=0.3)

# MAE
ax2.plot(epochs_range, history_base["val_mae"],  label="SimpleCNN val",  linestyle="--")
ax2.plot(epochs_range, history_mejor["val_mae"], label="MejorCNN val",   linestyle="-")
ax2.set_title("Val MAE (años)"); ax2.set_xlabel("Época"); ax2.set_ylabel("MAE (años)")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Evaluación final en Test Set

> El test set se usa **una sola vez** al final.  
> Usarlo durante el desarrollo introduce sesgo.

In [ ]:
def evaluate_test(model, model_path, test_loader, device=DEVICE):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    all_preds, all_targets = [], []

    with torch.inference_mode():
        for imgs, ages in test_loader:
            imgs = imgs.to(device)
            preds = model(imgs).squeeze(1).cpu()
            all_preds.extend(preds.tolist())
            all_targets.extend(ages.tolist())

    preds   = np.array(all_preds)
    targets = np.array(all_targets)
    mae  = np.abs(preds - targets).mean()
    rmse = np.sqrt(((preds - targets) ** 2).mean())
    return mae, rmse, preds, targets


base_mae, base_rmse, _, _ = evaluate_test(SimpleCNN().to(DEVICE), "mejor_simple.pth", test_loader)
mej_mae,  mej_rmse, mej_preds, mej_targets = evaluate_test(MejorCNN().to(DEVICE),  "mejor_mejor.pth",  test_loader)

print(f"{'Modelo':20s} {'MAE (años)':>12} {'RMSE (años)':>12}")
print("-" * 46)
print(f"{'SimpleCNN (baseline)':20s} {base_mae:>12.2f} {base_rmse:>12.2f}")
print(f"{'MejorCNN (tuya)':20s} {mej_mae:>12.2f} {mej_rmse:>12.2f}")
print(f"\nMejora en MAE: {base_mae - mej_mae:+.2f} años")

In [ ]:
# ── Scatter y distribución de errores (MejorCNN) ──────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"MejorCNN en Test Set — MAE = {mej_mae:.2f} años", fontsize=13)

# Scatter: predicción vs real
ax1.scatter(mej_targets, mej_preds, alpha=0.3, s=8, color="steelblue")
mn, mx = int(min(mej_targets)), int(max(mej_targets))
ax1.plot([mn, mx], [mn, mx], 'r--', label="Predicción perfecta")
ax1.set_xlabel("Edad real"); ax1.set_ylabel("Edad predicha")
ax1.set_title("Predicción vs Real")
ax1.legend(); ax1.grid(True, alpha=0.3)

# Histograma de errores
errors = mej_preds - mej_targets
ax2.hist(errors, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
ax2.axvline(0, color='red', linestyle='--', label="Error cero")
ax2.set_xlabel("Error (predicho − real)"); ax2.set_ylabel("Frecuencia")
ax2.set_title("Distribución de errores")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 10. Diagnóstico: overfitting / underfitting

In [ ]:
train_mae_final = history_mejor["train_mae"][-1]
val_mae_final   = history_mejor["val_mae"][-1]
gap = val_mae_final - train_mae_final

print("Diagnóstico final (MejorCNN):")
print(f"  Train MAE : {train_mae_final:.2f} años")
print(f"  Val MAE   : {val_mae_final:.2f} años")
print(f"  Gap       : {gap:.2f} años", end="  ")

if gap > 5:
    print("→ Posible OVERFITTING (considera más Dropout, menos capacidad o más augmentations)")
elif train_mae_final > 10:
    print("→ Posible UNDERFITTING (considera más capas, más filtros o más épocas)")
else:
    print("→ Buen balance")

---
## 11. Preguntas de reflexión y tabla de experimentos

1. ¿Cuánta mejora en MAE obtuviste al pasar de SimpleCNN (3 bloques) a MejorCNN (5 bloques)?

2. ¿Para qué sirve `BatchNorm2d`? ¿Notaste alguna diferencia en la estabilidad del entrenamiento?

3. Mira el scatter plot. ¿Para qué rangos de edad el modelo es más preciso? ¿A qué se debe?

4. **Experimenta** y registra los resultados en la tabla:

| Cambio realizado | Val MAE | Test MAE |
|---|---|---|
| SimpleCNN baseline (3 bloques) | | |
| + 2 bloques convolucionales | | |
| + BatchNorm en cada bloque | | |
| + Dropout(0.4) en el regresor | | |
| + más augmentations en train | | |

5. ¿Hay un punto donde agregar más bloques deja de mejorar o empeora? ¿Por qué?